In [27]:
# Imports

import pandas as pd
import duckdb

In [1]:
# Load CSV

df = pd.read_csv("retail_store_inventory.csv")


NameError: name 'pd' is not defined

In [29]:
# Fix Date Type

df['Date'] = pd.to_datetime(df['Date'])

In [30]:
# Create Database Connection

con = duckdb.connect("inventory_flow.db")

In [31]:
# Register DataFrame

con.register("inventory_df", df)

In [42]:
# Create Raw Table

con.execute("""

CREATE OR REPLACE TABLE retail_store_inventory AS

SELECT *
FROM inventory_df

""")

In [43]:
# Verify Raw Data

con.sql("""
SELECT COUNT(*)
FROM retail_store_inventory
""").df()

,count_star()
0,73100


In [44]:
# Build Dimensions
# Dim Date

con.execute("""

CREATE OR REPLACE TABLE dim_date AS

SELECT DISTINCT

CAST(
strftime(Date,'%Y%m%d')
AS INTEGER
) AS date_key,

Date AS full_date,

EXTRACT(YEAR FROM Date) AS year,
EXTRACT(QUARTER FROM Date) AS quarter,
EXTRACT(MONTH FROM Date) AS month,
EXTRACT(WEEK FROM Date) AS week,
EXTRACT(DAY FROM Date) AS day

FROM retail_store_inventory

""")

In [45]:
# Store Dimension

con.execute("""

CREATE OR REPLACE TABLE dim_store AS

SELECT

ROW_NUMBER()
OVER(
ORDER BY store_id, region
)
AS store_key,

store_id,
region

FROM (

SELECT DISTINCT

"Store ID" AS store_id,
Region AS region

FROM retail_store_inventory

)

""")

In [46]:
# Product Dimension

con.execute("""

CREATE OR REPLACE TABLE dim_product AS

SELECT

ROW_NUMBER()
OVER(
ORDER BY product_id
)
AS product_key,

product_id,
category

FROM (

SELECT DISTINCT

"Product ID" AS product_id,
Category AS category

FROM retail_store_inventory

)

""")

In [47]:
# Verify Store Counts

con.sql("""

SELECT
COUNT(*) AS total_rows,
COUNT(DISTINCT store_id)
AS unique_stores

FROM dim_store

""").df()


,total_rows,unique_stores
0,20,5


In [48]:
#Verify Product Counts

con.sql("""

SELECT
COUNT(*) AS total_rows,
COUNT(DISTINCT product_id)
AS unique_products

FROM dim_product

""").df()

,total_rows,unique_products
0,100,20


In [49]:
con.sql("""

SELECT *
FROM dim_product
LIMIT 20

""").df()

,product_key,product_id,category
0,1,P0001,Furniture
1,2,P0001,Clothing
2,3,P0001,Groceries
3,4,P0001,Electronics
4,5,P0001,Toys
5,6,P0002,Electronics
6,7,P0002,Furniture
7,8,P0002,Groceries
8,9,P0002,Toys
9,10,P0002,Clothing


In [50]:
# Rebuild Product Dimension

con.execute("""
DROP TABLE IF EXISTS dim_product
""")

In [51]:
con.execute("""

CREATE OR REPLACE TABLE dim_product AS

SELECT

ROW_NUMBER()
OVER(
ORDER BY product_id, category
)
AS product_key,

product_id,
category

FROM (

SELECT DISTINCT

"Product ID" AS product_id,
Category AS category

FROM retail_store_inventory

)

""")

In [52]:
con.sql("""

SELECT
COUNT(*) AS rows,
COUNT(DISTINCT product_id || category)
AS unique_product_category

FROM dim_product

""").df()

,rows,unique_product_category
0,100,100


In [53]:
con.sql("""

SELECT COUNT(*) AS join_count

FROM retail_store_inventory r

JOIN dim_store s

ON r."Store ID" = s.store_id
AND r.Region = s.region

JOIN dim_product p

ON r."Product ID" = p.product_id
AND r.Category = p.category

""").df()


,join_count
0,73100


In [54]:
# Drop Bad Product Dimension

con.execute("""
DROP TABLE IF EXISTS dim_product
""")

In [55]:
# Rebuild Product Dimension

con.execute("""

CREATE OR REPLACE TABLE dim_product AS

SELECT

ROW_NUMBER()
OVER(
ORDER BY product_id, category
)
AS product_key,

product_id,
category

FROM (

SELECT DISTINCT

"Product ID" AS product_id,
Category AS category

FROM retail_store_inventory

)

""")

In [56]:
# Verify Product Dimension

con.sql("""

SELECT
COUNT(*) AS rows,
COUNT(DISTINCT product_id || category)
AS unique_product_category

FROM dim_product

""").df()

,rows,unique_product_category
0,100,100


In [57]:
# Verify Correct Join Count

con.sql("""

SELECT COUNT(*) AS join_count

FROM retail_store_inventory r

JOIN dim_store s
ON r."Store ID" = s.store_id
AND r.Region = s.region

JOIN dim_product p
ON r."Product ID" = p.product_id
AND r.Category = p.category

""").df()


,join_count
0,73100


In [58]:
# Remove Old Fact Table

con.execute("""
DROP TABLE IF EXISTS fact_inventory
""")

In [59]:
# Create Fact Table

con.execute("""

CREATE OR REPLACE TABLE fact_inventory AS

SELECT

d.date_key,

s.store_key,

p.product_key,

r."Weather Condition" AS weather_condition,

r."Holiday/Promotion" AS holiday_promotion,

r.Seasonality,

r."Inventory Level" AS inventory_level,

r."Units Sold" AS units_sold,

r."Units Ordered" AS units_ordered,

r."Demand Forecast" AS demand_forecast,

r.Price,

r.Discount AS discount_pct,

r."Competitor Pricing" AS competitor_price,

r."Units Sold" * r.Price
AS revenue,

r."Units Sold" - r."Demand Forecast"
AS forecast_error,

CASE
WHEN r."Inventory Level" = 0
THEN 0
ELSE
CAST(r."Units Sold" AS DOUBLE)
/
CAST(r."Inventory Level" AS DOUBLE)
END
AS inventory_velocity,

CASE
WHEN r."Demand Forecast"
     >
     r."Inventory Level"
THEN 1
ELSE 0
END
AS stockout_flag,

CASE
WHEN r."Inventory Level"
     >
     r."Demand Forecast" * 1.5
THEN 1
ELSE 0
END
AS overstock_flag

FROM retail_store_inventory r

JOIN dim_date d
ON r.Date = d.full_date

JOIN dim_store s
ON r."Store ID" = s.store_id
AND r.Region = s.region

JOIN dim_product p
ON r."Product ID" = p.product_id
AND r.Category = p.category

""")

In [60]:
# Verify Fact Table

con.sql("""

SELECT COUNT(*) AS fact_rows

FROM fact_inventory

""").df()

,fact_rows
0,73100


In [61]:
con.sql("""

SELECT *

FROM fact_inventory

LIMIT 10

""").df()

,date_key,store_key,product_key,weather_condition,holiday_promotion,Seasonality,inventory_level,units_sold,units_ordered,demand_forecast,Price,discount_pct,competitor_price,revenue,forecast_error,inventory_velocity,stockout_flag,overstock_flag
0,20220101,2,4,Rainy,0,Autumn,231,127,55,135.47,33.50,20,29.69,4254.50,-8.47,0.549784,0,1
1,20220101,3,10,Sunny,0,Autumn,204,150,66,144.04,63.01,20,66.16,9451.50,5.96,0.735294,0,0
2,20220101,4,15,Sunny,1,Summer,102,65,51,74.02,27.99,10,31.32,1819.35,-9.02,0.637255,0,0
3,20220101,2,20,Cloudy,1,Autumn,469,61,164,62.18,32.72,10,34.74,1995.92,-1.18,0.130064,0,1
4,20220101,1,22,Sunny,0,Summer,166,14,135,9.26,73.64,0,68.95,1030.96,4.74,0.084337,0,1
5,20220101,3,29,Sunny,1,Winter,138,128,102,139.82,76.83,10,79.35,9834.24,-11.82,0.927536,1,0
6,20220101,1,33,Rainy,1,Winter,359,97,167,108.92,34.16,10,36.55,3313.52,-11.92,0.270195,0,1
7,20220101,2,36,Cloudy,0,Spring,380,312,54,329.73,97.99,5,100.09,30572.88,-17.73,0.821053,0,0
8,20220101,4,42,Cloudy,0,Autumn,183,175,135,174.15,20.74,10,17.66,3629.50,0.85,0.956284,0,0
9,20220101,3,50,Rainy,1,Winter,108,28,196,24.47,59.99,0,61.21,1679.72,3.53,0.259259,0,1


In [63]:
# Executive KPI View

con.execute("""

CREATE OR REPLACE VIEW vw_executive_kpis AS

SELECT

SUM(inventory_level * price)
AS inventory_value,

SUM(units_sold)
/ AVG(inventory_level)
AS inventory_turnover,

AVG(

CASE

WHEN units_sold = 0
THEN 0

ELSE

1 -
ABS(
units_sold - demand_forecast
)
/
units_sold

END

)

AS forecast_accuracy,

COUNT(

CASE

WHEN stockout_flag = 1
THEN 1

END

)

AS predicted_stockouts,

COUNT(

CASE

WHEN overstock_flag = 1
THEN 1

END

)

AS overstocked_skus,

SUM(revenue)
AS total_revenue,

AVG(inventory_velocity)
AS avg_inventory_velocity

FROM fact_inventory

""")

In [64]:
# View KPIs

con.sql("""

SELECT *

FROM vw_executive_kpis

""").df()

,inventory_value,inventory_turnover,forecast_accuracy,predicted_stockouts,overstocked_skus,total_revenue,avg_inventory_velocity
0,1.108476e+09,36344.906455,0.749057,2585,47085,5.502289e+08,0.498005


In [65]:
# Revenue At Risk

con.execute("""

CREATE OR REPLACE VIEW vw_revenue_at_risk AS

SELECT

SUM(
demand_forecast * price
)
AS revenue_at_risk

FROM fact_inventory

WHERE stockout_flag = 1

""")

In [66]:
con.sql("""

SELECT *

FROM vw_revenue_at_risk

""").df()

,revenue_at_risk
0,2.875514e+07


In [69]:
# Inventory Health Score

con.execute("""

CREATE OR REPLACE VIEW vw_inventory_health AS

SELECT

AVG(

    (
        CASE
            WHEN stockout_flag = 0 THEN 100
            ELSE 0
        END
    ) * 0.40

    +

    (
        CASE
            WHEN units_sold = 0 THEN 0
            ELSE

            GREATEST(

                0,

                (
                    1 -
                    ABS(
                        units_sold - demand_forecast
                    )
                    /
                    units_sold
                ) * 100

            )

        END
    ) * 0.30

    +

    LEAST(
        inventory_velocity * 100,
        100
    ) * 0.20

    +

    (
        CASE
            WHEN stockout_flag = 0 THEN 100
            ELSE 0
        END
    ) * 0.10

)

AS inventory_health_score

FROM fact_inventory

""")

In [70]:
con.sql("""

SELECT *

FROM vw_inventory_health

""").df()

,inventory_health_score
0,83.39052


In [72]:
con.close()